[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/geoai/blob/main/docs/examples/visualization.ipynb)

# Visualize Vector and Raster Data with GeoAI

GeoAI provides high-level helpers and interactive map classes for inspecting rasters, vector layers, and side-by-side comparisons in Jupyter notebooks.

| Function / Class | Purpose | Output |
| --- | --- | --- |
| `geoai.view_raster` | Display a local GeoTIFF or remote Cloud Optimized GeoTIFF (COG) | Interactive map (`folium` or `ipyleaflet`) |
| `geoai.view_vector_interactive` | Display vector data with attribute styling and optional raster backdrop | Interactive map (`folium`) |
| `geoai.create_split_map` | Compare two basemaps, rasters, or vector and raster layers with a swipe slider | Interactive split map (`ipyleaflet`) |
| `geoai.LeafMap` | Compose custom multi-layer maps, basemaps, and legends | Interactive map (`ipyleaflet`) |
| `geoai.Map` | Render 2D and 3D globe maps with GPU-accelerated vector and COG layers | Interactive map (`MapLibre GL JS`) |
| `geoai.view_image` | Quick raster or tensor preview with percentile contrast stretch | Static Matplotlib figure |
| `geoai.view_vector` | Static choropleth map or vector outline overlay on a raster | Static Matplotlib figure |
| `geoai.display_image_with_vector` | Side-by-side comparison of raw imagery and imagery with vector overlay | Static Matplotlib figure |

In [ ]:
# %pip install -U geoai-py

In [ ]:
import geoai

cities_url = "https://data.source.coop/giswqs/opengeos/world_cities.geojson"
nlcd_url = "https://data.source.coop/giswqs/opengeos/nlcd_2021_land_cover_30m.tif"
naip_url = (
    "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/naip_rgb_train.tif"
)
buildings_url = "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/naip_train_buildings.geojson"
masks_url = "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/naip_building_masks.tif"

## Visualize remote raster data

`geoai.view_raster` streams Cloud Optimized GeoTIFFs directly over HTTP without downloading the full file. By default it uses the `folium` backend and adds layer and opacity controls.

In [ ]:
geoai.view_raster(nlcd_url, layer_name="NLCD 2021", opacity=0.8)

Display high-resolution NAIP RGB aerial imagery directly from its remote URL.

In [ ]:
geoai.view_raster(naip_url, layer_name="NAIP Imagery")

Pass `backend="ipyleaflet"` to use an `ipyleaflet` widget and supply another raster URL via `basemap` to overlay a prediction or mask raster on top of imagery.

In [ ]:
geoai.view_raster(
    masks_url,
    colormap="blues",
    opacity=0.6,
    basemap=naip_url,
    layer_name="Building Masks",
    backend="ipyleaflet",
)

## Visualize vector data interactively

`geoai.view_vector_interactive` accepts a URL, a local file path (GeoJSON, Shapefile, GeoPackage, GeoParquet), or a `GeoDataFrame`.

In [ ]:
geoai.view_vector_interactive(cities_url, layer_name="World Cities")

Pass `column` to style vector features as a choropleth map and `tiles` to place a remote or local raster underneath the vector layer.

In [ ]:
geoai.view_vector_interactive(
    buildings_url,
    column="height",
    cmap="viridis",
    tiles=naip_url,
    layer_name="Building Heights",
)

You can also apply a uniform custom style with `style_kwds` when overlaying footprints on imagery.

In [ ]:
geoai.view_vector_interactive(
    buildings_url,
    style_kwds={"color": "#ef4444", "fillOpacity": 0.4},
    tiles=naip_url,
    layer_name="Buildings",
)

## Create split-panel comparison maps

`geoai.create_split_map` creates an interactive swipe slider to compare basemaps, rasters, or vector overlays side by side.

### Compare two basemaps

In [ ]:
geoai.create_split_map(
    left_layer="SATELLITE",
    right_layer="OpenStreetMap",
    left_label="Satellite",
    right_label="OpenStreetMap",
    center=[47.653, -117.598],
    zoom=15,
)

### Compare a raster mask with aerial imagery

In [ ]:
geoai.create_split_map(
    left_layer=masks_url,
    right_layer=naip_url,
    left_label="Building Masks",
    right_label="NAIP Imagery",
    left_args={"colormap": {"255": "#2563eb"}, "opacity": 0.6},
    basemap=naip_url,
)

### Compare vector polygons with aerial imagery

In [ ]:
geoai.create_split_map(
    left_layer=buildings_url,
    right_layer=naip_url,
    left_label="Buildings",
    right_label="NAIP Imagery",
    left_args={"style": {"color": "#ef4444", "fillOpacity": 0.4}},
    basemap=naip_url,
)

## Combine multiple layers with `geoai.LeafMap`

`geoai.LeafMap` subclasses `leafmap.Map` (backed by `ipyleaflet`) for composing basemaps, COGs, and vector overlays in a single interactive canvas.

In [ ]:
m = geoai.LeafMap(center=[40, -100], zoom=4)
m.add_basemap("Esri.WorldImagery")
m.add_cog_layer(nlcd_url, name="NLCD 2021", opacity=0.75)
m.add_geojson(cities_url, layer_name="World Cities")
m

## 2D and 3D visualization with `geoai.Map` (MapLibre)

`geoai.Map` subclasses `leafmap.maplibregl.Map` for WebGL-accelerated rendering, custom paint expressions, 3D globe projection, and standalone HTML export.

In [ ]:
m = geoai.Map(center=[-117.598, 47.653], zoom=15, style="liberty")
m.add_cog_layer(naip_url, name="NAIP")
m.add_geojson(
    buildings_url,
    name="Buildings",
    layer_type="fill",
    paint={
        "fill-color": "#ef4444",
        "fill-opacity": 0.4,
        "fill-outline-color": "#facc15",
    },
)
m.add_layer_control()
m

Set `projection="globe"` to render global vector and raster layers on a 3D globe.

In [ ]:
m_globe = geoai.Map(
    center=[-100, 40], zoom=3, style="positron", projection="globe"
)
m_globe.add_cog_layer(nlcd_url, name="NLCD 2021", opacity=0.8)
m_globe.add_geojson(
    cities_url,
    name="World Cities",
    layer_type="circle",
    paint={"circle-radius": 3, "circle-color": "#ef4444"},
)
m_globe.add_layer_control()
m_globe

Export the MapLibre map to a self-contained HTML file that can be opened in any browser.

In [ ]:
m.to_html("geoai_visualization_map.html", title="GeoAI Visualization")

## Visualize local raster and vector files

Download local copies of the sample NAIP raster and building footprints to demonstrate local interactive viewing and static Matplotlib figures.

In [ ]:
naip_path = geoai.download_file(naip_url)
buildings_path = geoai.download_file(buildings_url)

### Interactive map of a local GeoTIFF

`geoai.view_raster` also serves local GeoTIFF files via `localtileserver`.

In [ ]:
geoai.view_raster(naip_path, layer_name="Local NAIP")

### Quick static raster preview with `geoai.view_image`

In [ ]:
geoai.view_image(naip_path, title="NAIP Imagery")

### Static vector overlay and choropleth with `geoai.view_vector`

Set `raster_path` and `outline_only=True` to draw vector outlines directly over a local raster, automatically reprojecting the vector layer if the CRS differs.

In [ ]:
geoai.view_vector(
    buildings_path,
    raster_path=naip_path,
    outline_only=True,
    edge_color="yellow",
    outline_linewidth=1.5,
    title="Building Footprints over NAIP Imagery",
)

Pass `column` and `classification` to create a static choropleth map of a numeric attribute.

In [ ]:
geoai.view_vector(
    buildings_path,
    column="height",
    cmap="viridis",
    classification="quantiles",
    title="Building Heights",
)

### Side-by-side image and vector overlay with `geoai.display_image_with_vector`

In [ ]:
fig, axes, info = geoai.display_image_with_vector(
    naip_path,
    buildings_path,
    vector_color="yellow",
    vector_linewidth=1.5,
)
print(f"Features: {info['num_features']}, CRS: {info['image_crs']}")